##CSV파일읽기

In [ ]:
import pandas as pd
try:
    df_house = pd.read_csv("/content/1.01.house.price.csv") #1,2
except FileNotFoundError:                                 #5
    print("파일명과 업로드 여부를 확인하세요")

else:
    print("첫번째 행:")
    print(df_house.iloc[0])                                 #3
    print("전체 데이터 행 수:",len(df_house))               #3
    print("price 자료형:",type(df_house.iloc[0]["price"]))  #4


첫번째 행:
ID        1.0
size     34.0
year      5.0
price     5.0
Name: 0, dtype: float64
전체 데이터 행 수: 10
price 자료형: <class 'numpy.float64'>


##문자열을 숫자로 변환하기

In [ ]:
houses = []
skip_count = 0
for row_num, row in df_house.iterrows():
  if pd.isna(row["ID"])or pd.isna(row["size"]) or pd.isna(row["year"]) or pd.isna(row["price"]):
    print(f"{row_num}번 행 제외: 필수 값 누락" )
    skip_count += 1
    continue
  try:
    house = {
        "ID": int(row["ID"]),
        "size": int(row["size"]),
        "year": int(row["year"]),
        "price": float(row["price"]),
    }
  except (ValueError, TypeError) as e:
    print(f"{row_num}번 행 제외: 숫자로 변환 불가({e})")
    skip_count += 1
    continue
  houses.append(house)

print("정상 저장 행 수 :", len(houses))
print("제외 행 수", skip_count)
print("첫 번째 주택:", houses[0])

정상 저장 행 수 : 10
제외 행 수 0
첫 번째 주택: {'ID': 1, 'size': 34, 'year': 5, 'price': 5.0}


##조건에 맞는 주택 찾기

In [ ]:
house_test = houses.copy()
house_test.append({"ID": 101, "year": 5, "price": 3})  # size 키 없음
house_test.append({"ID": 102, "size": 35, "year": 5, "price": None})  # 값 없음
house_test.append({"ID": 103, "size": "오류", "year": 5, "price": 3})  # 숫자로 변환할 수 없음
house_test.append({"ID": 104, "size": 35, "year": 5, "price": 3}) # 정상 데이터
selected = []

for house in house_test:

  #키 값 누락 확인하기
  if "size" not in house or "price" not in house:
    print(f"ID: {house["ID"]}제외: size 또는 Price 값 없음")
    continue

  if house["size"] is None or  house["price"] is None or pd.isna(house["size"]) or pd.isna(house["price"]):
   # is None과 pd.isna를 둘 다 쓰는 이유
   # None: 파이썬의 '값 없음'
   # NaN: pandas/숫자 세계의 '값 없음' -> isna 만 써도 되긴함
    print(f"ID: {house['ID']}제외: size 또는 price 값 없음(None/결측)")
    continue


  try:
     size = float(house["size"])
     price = float(house["price"])
  except(ValueError, TypeError):
    print(f"ID: {house["ID"]}제외: 숫자로 변환할 수 없는 값 ")
    continue


  if size <= 0 or price<0:
      print(f"ID {house["ID"]}제외: size가 0이하이거나 price가 음수")
      continue
  if size>=30 and price<=4:
      selected.append(house)


print("---선택된 주택 ID ---")
for house in selected:
  print(house["ID"])

print("선택된 주택 수:", len(selected))
print("원본 houses행 수(변화없음):", len(houses))


ID: 101제외: size 또는 Price 값 없음
ID: 102제외: size 또는 price 값 없음(None/결측)
ID: 103제외: 숫자로 변환할 수 없는 값 
---선택된 주택 ID ---
3
4
5
9
10
104
선택된 주택 수: 6
원본 houses행 수(변화없음): 10


## 함수와 예외 처리

In [ ]:
def unit_price(price, size):
    try:

        price = float(house["price"])
        size = float(house["size"])

    except(ValueError):
        print("값오류: 숫자로 변환할 수 없는 문자열입니다")
        return None
    except(TypeError):
        print("자료형 오류: 숫자또는 숫자 형태의 문자열을 입력하세요")
        return None
    if size<=0:
        print("크기오류")
        return None
    if price<0:
        print("가격오류")
        return None
    return price/size
for house in houses:
    result = unit_price(house["price"], house["size"])
    if result is not None:
        print(f"ID: {house["ID"]}: 크기당 가격 {result:.4f}")

ID: 1: 크기당 가격 0.1471
ID: 2: 크기당 가격 0.1000
ID: 3: 크기당 가격 0.1333
ID: 4: 크기당 가격 0.0789
ID: 5: 크기당 가격 0.0750
ID: 6: 크기당 가격 0.0875
ID: 7: 크기당 가격 0.0885
ID: 8: 크기당 가격 0.1000
ID: 9: 크기당 가격 0.0882
ID: 10: 크기당 가격 0.0853


##목적에 맞는 자료구조 선택하기

In [ ]:
# 1. 전체 주택 ID를 원래 순서대로 저장
# List 선택
# 이유: 순서 유지가 가능하고 차레대로 추가할 수 있다.
id_list = []
for house in houses:
    id_list.append(house["ID"])
print("ID목록:", id_list)

# 2. year 값을 중복없이 저장
# 집합(set)선택
# 이유: 집합은 같은 값은 한번만 저장함. 중복없이 저장가능
year_set = set()
for house in houses:
    year_set.add(house["year"])
print("중복없는 year:", year_set)

# 3. 주택 ID로 가격을 조회,  ID 3의 가격을 출력
# 딕셔너리 선택
# 이유: 키로 값을 바로 꺼낼 수 있다.
price_by_id = {}
for house in houses:
    price_by_id[house["ID"]] = house["price"]
print("ID 3 장의 가격:", price_by_id[3])

# 4.  최소 크기 30과 최대 가격 4를 각각 하나의 원소로 갖는 묶음으로 저장
# 튜플 선택
# 이유: 튜플은 생성 뒤 추가/삭제 등이 불가함
search_condition = (30,4)
print("검색 조건(최소크기, 최대크기, 최대가격)", search_condition)

ID목록: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
중복없는 year: {2, 5, 12, 18, 19, 20, 22}
ID 3 장의 가격: 4.0
검색 조건(최소크기, 최대크기, 최대가격) (30, 4)
